In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import io
warnings.filterwarnings('ignore')

from google.colab import files

#Story 1: The schema Builder

# Load the raw CSVs

print('Upload olist_orders_dataset.csv')
uploaded_orders = files.upload()
orders_filename = list(uploaded_orders.keys())[0]
orders = pd.read_csv (io.BytesIO(uploaded_orders[orders_filename]), compression='zip')
print(f'Orders loaded:{orders.shape} rows')

print('upload olist_order_reviews_dataset.csv')
uploaded_reviews = files.upload()
reviews_filename = list(uploaded_reviews.keys())[0]
reviews = pd.read_csv (io.BytesIO(uploaded_reviews[reviews_filename]), compression='zip')
print(f'Reviews loaded: {reviews.shape} rows')

print('Upload olist_customers_dataset.csv')
uploaded_customers = files.upload()
customers_filename = list(uploaded_customers.keys())[0]
customers = pd.read_csv (io.BytesIO(uploaded_customers[customers_filename]), compression='zip')
print(f'Customers loaded:{customers.shape} rows')

#perform correct joins

merged = orders.merge(reviews, on='order_id', how='left')
master = merged.merge(customers, on='customer_id', how='left')

# Duplicate check and handling (keep most recent review)
master_original_count = len(master)
master = master.sort_values('review_creation_date').drop_duplicates(subset=['order_id'], keep='last')
print(f"Removed {master_original_count - len(master):,} duplicate rows | {len(master):,} unique orders")

#keep only deliverd orders

delivered = master[master['order_status'] == 'delivered'].copy()

print(f' Master dataset: {len(master):,} rows')
print(f' Delivered orders: {len(delivered):,} rows')

#Story 2: The "Real" Delay Calculator

# Convert dates to datetime
delivered['order_estimated_delivery_date'] = pd.to_datetime(delivered['order_estimated_delivery_date'])
delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])

# Remove rows with missing dates
delivered = delivered.dropna(subset=['order_estimated_delivery_date', 'order_delivered_customer_date'])

# Calculate Days_Difference (positive = early delivery, negative = late delivery)
delivered['Days_Difference'] = (delivered['order_estimated_delivery_date'] - delivered['order_delivered_customer_date']).dt.days

# Classify delays
delivered['delivery_status'] = delivered['Days_Difference'].apply(
    lambda x: 'On Time' if x >= 0 else 'Late (1-5 days)' if x >= -5 else 'Super Late'
)

# STORY 3: Geographic Heatmap

# Calculation of % of late orders per state
state_analysis = delivered.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_orders=('Days_Difference', lambda x: (x < 0).sum())
).reset_index()

state_analysis['late_percentage'] = (state_analysis['late_orders'] / state_analysis['total_orders']) * 100

# visualize on dashboard
state_analysis.to_csv('state_analysis.csv', index=False)

# STORY 4: Sentiment Correlation

# Calculation of average review score by delivery delay status
sentiment_analysis = delivered.groupby('delivery_status')['review_score'].agg(
    avg_review='mean',
    order_count='count'
).reset_index()

# Calculation of correlation between delay days and review score
correlation = delivered['Days_Difference'].corr(delivered['review_score'])

# visualization for dashboard
sentiment_analysis.to_csv('sentiment_analysis.csv', index=False)

# Save main data for dashboard
dashboard_data = delivered[['order_id', 'customer_state', 'customer_city', 'review_score',
                             'Days_Difference', 'delivery_status']].copy()
dashboard_data.to_csv('master_delivery_data.csv', index=False)

# Download all files
print("\nDownloading files for dashboard...")
files.download('master_delivery_data.csv')
files.download('state_analysis.csv')
files.download('sentiment_analysis.csv')

# Story 5: Product translation

# upload datas
print('Upload product_category_name_translation.csv')
uploaded_trans = files.upload()
translation = pd.read_csv(io.BytesIO(list(uploaded_trans.values())[0]))

print('Upload olist_products_dataset.csv')
uploaded_products = files.upload()
products = pd.read_csv(io.BytesIO(list(uploaded_products.values())[0]), compression='zip')

# Load order_items dataset to get product_id, needed for merging with products
print('Upload olist_order_items_dataset.csv')
uploaded_items_for_products = files.upload()
order_items = pd.read_csv(io.BytesIO(list(uploaded_items_for_products.values())[0]), compression='zip')

# Merge to get English categories (from delivered instead of master)

# merge delivered with order_items to get product_id
delivered = delivered.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')

# merge with products to get product_category_name
delivered = delivered.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')

# merge with translation to get english category names
delivered = delivered.merge(translation, on='product_category_name', how='left')
print("Translation complete:")
print(translation[translation['product_category_name'] == 'cama_mesa_banho'])


# Strory 6: Candidate's choice (Order Accuracy Analysis)

# Load data
print('Upload olist_sellers_dataset.csv')
uploaded_sellers = files.upload()
sellers = pd.read_csv(io.BytesIO(list(uploaded_sellers.values())[0]))

# order_items is already loaded in Story 5

# Connect orders to sellers
seller_orders = delivered.merge(order_items, on='order_id').merge(sellers, on='seller_id')

# Calculate seller performance
seller_perf = seller_orders.groupby('seller_id').agg(
    state=('seller_state', 'first'),
    orders=('order_id', 'count'),
    late_pct=('Days_Difference', lambda x: (x < 0).mean() * 100)
).reset_index()

# Keep sellers with at least 10 orders
seller_perf = seller_perf[seller_perf['orders'] >= 10].sort_values('late_pct', ascending=False)


print(f"Top worst seller: {seller_perf.iloc[0]['seller_id']} ({seller_perf.iloc[0]['late_pct']:.1f}% late)")

print("\n Complete! 7 CSV files downloaded for dashboard")


Upload olist_orders_dataset.csv


Saving olist_orders_dataset.csv.zip to olist_orders_dataset.csv.zip
Orders loaded:(99441, 8) rows
upload olist_order_reviews_dataset.csv


Saving olist_order_reviews_dataset.csv.zip to olist_order_reviews_dataset.csv.zip
Reviews loaded: (99224, 7) rows
Upload olist_customers_dataset.csv


Saving olist_customers_dataset.csv.zip to olist_customers_dataset.csv.zip
Customers loaded:(99441, 5) rows
Removed 551 duplicate rows | 99,441 unique orders
 Master dataset: 99,441 rows
 Delivered orders: 96,478 rows



<IPython.core.display.Javascript object>

...

Upload product_category_name_translation.csv


Saving product_category_name_translation.csv to product_category_name_translation.csv
Upload olist_products_dataset.csv


Saving olist_products_dataset.csv.zip to olist_products_dataset.csv.zip
Upload olist_order_items_dataset.csv


Saving olist_order_items_dataset.csv.zip to olist_order_items_dataset.csv.zip
Translation complete:
  product_category_name product_category_name_english
3       cama_mesa_banho                bed_bath_table
Upload olist_sellers_dataset.csv


Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Top worst seller: 8d92f3ea807b89465643c219455e7369 (97.4% late)

 Complete! 7 CSV files downloaded for dashboard
